# 06b_svo_networks — Directed SVO networks

> **Environment:** requires the project venv **`.venv311`** (Python 3.11) as the Jupyter kernel — the pipeline dependencies are installed only there. `run_all.command` uses it automatically. See `README.md` → Environment setup.

**Input:** `data/output/edges/svo_edges_*.jsonl` (from 05b)
**Output:**
- `data/output/networks/G_svo_{window}.graphml` — directed actor→actor graph (Gephi/Cytoscape/igraph)
- `data/output/networks/G_svo_{window}.gexf` — Gephi-preferred
- `data/output/networks/G_svo_{window}.html` — pyvis preview (directed arrows)

The SVO-track parallel of `06_networks`. Where `06_networks` builds an **undirected bipartite** actor–concept graph per window, this builds a **directed unipartite** actor→actor graph: one `nx.DiGraph` per window, edges `subject → object` typed by CAMEO quadrant and polarity (the shared `POLARITY_COLOR` scheme). All whitelisted actors are added as nodes (isolated = no SVO relation that window), so node sets are comparable across windows for 07b; the pyvis preview renders only connected actors for readability.

There is **no bipartite / projection step** — the SVO graph is already the directed, semantically-grounded analogue of `06`'s actor–actor projection (real subject→object relations, not co-membership).

## Pipeline steps in this notebook

1. Setup & paths
2. Load svo_edges files
3. Build one directed graph per window (edge attrs: weight, CAMEO, polarity, verbs; node attrs: in/out degree & weight, entity_type)
4. Validation report — agents (out-weight) vs targets (in-weight), top directed relations
5. Save GraphML
6. Save GEXF
7. pyvis interactive HTML — directed arrows, edges coloured by polarity, with legend

## Step 1: Setup & paths

In [ ]:
import json
import sys
from pathlib import Path
from collections import defaultdict

import networkx as nx

_cwd = Path().resolve()
ROOT = next(
    (p for p in [_cwd] + list(_cwd.parents) if (p / 'src').is_dir()),
    _cwd,
)
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from src.alias_map    import ACTOR_WHITELIST, ACTOR_TYPE
from src.concept_dict import POLARITY_COLOR
from src.time_windows import TIME_WINDOWS, window_file_label

EDGES_DIR    = ROOT / 'data' / 'output' / 'edges'
NETWORKS_DIR = ROOT / 'data' / 'output' / 'networks'
NETWORKS_DIR.mkdir(parents=True, exist_ok=True)

# Typed + per-format layout with chronological filename prefixes (see 06).
for _sub in ('svo/graphml', 'svo/gexf', 'svo/html'):
    (NETWORKS_DIR / _sub).mkdir(parents=True, exist_ok=True)

WINDOW_ORDER    = [w[0] for w in TIME_WINDOWS]
CAMEO_QUADRANTS = ['material_conflict', 'verbal_conflict',
                   'verbal_cooperation', 'material_cooperation']

print(f'Edges dir    : {EDGES_DIR}')
print(f'Networks dir : {NETWORKS_DIR}')
print(f'Actors in whitelist : {len(ACTOR_WHITELIST)}')

## Step 2: Load svo_edges files

In [ ]:
edge_files = sorted(EDGES_DIR.glob('svo_edges_*.jsonl'))
print(f'Found {len(edge_files)} svo_edges file(s):')
for f in edge_files:
    print(f'  {f.name}')
assert edge_files, 'No svo_edges files — run 05b_svo_edges first'

svo_by_window = defaultdict(list)
for f in edge_files:
    with open(f, encoding='utf-8') as fh:
        for line in fh:
            e = json.loads(line)
            svo_by_window[e['window']].append(e)

print('\nDirected edges loaded per window:')
for w in WINDOW_ORDER:
    if w in svo_by_window:
        print(f'  {w:20s}  {len(svo_by_window[w])} edges')

## Step 3: Build directed graphs

One `nx.DiGraph` per window. All whitelisted actors are added as nodes (isolated where they had no SVO relation). Edge attributes: `weight`, `weight_normalized`, `dominant_cameo`, `dominant_polarity`, the per-quadrant `cameo_*` counts, `polarity_positive/negative`, `top_verb` (+ `n_verbs`), `top_source` (+ `n_sources`). Node attributes: `node_type`, `entity_type`, and directed stats `in_degree` / `out_degree` / `in_weight` / `out_weight` (out = **agent/aggressor**, in = **target/patient**).

In [ ]:
graphs = {}
for window in WINDOW_ORDER:
    if window not in svo_by_window:
        continue
    G = nx.DiGraph()
    G.graph['window'] = window

    for actor in sorted(ACTOR_WHITELIST):
        G.add_node(actor, node_type='actor',
                   entity_type=ACTOR_TYPE.get(actor, 'UNKNOWN'))

    for e in svo_by_window[window]:
        verbs   = e.get('verbs', {}) or {}
        sources = e.get('sources', {}) or {}
        attrs = {
            'weight':            int(e['weight']),
            'weight_normalized': float(e['weight_normalized']),
            'dominant_cameo':    e['dominant_cameo'],
            'dominant_polarity': e['dominant_polarity'],
            'polarity_positive': int(e.get('polarity_positive', 0)),
            'polarity_negative': int(e.get('polarity_negative', 0)),
            'top_verb':          next(iter(verbs), ''),
            'n_verbs':           len(verbs),
            'top_source':        max(sources.items(), key=lambda kv: kv[1])[0] if sources else '',
            'n_sources':         len(sources),
        }
        for q in CAMEO_QUADRANTS:
            attrs[f'cameo_{q}'] = int(e.get(f'cameo_{q}', 0))
        G.add_edge(e['subject'], e['object'], **attrs)

    # Directed node stats (out = agent, in = target)
    for n in G.nodes():
        G.nodes[n]['out_degree'] = G.out_degree(n)
        G.nodes[n]['in_degree']  = G.in_degree(n)
        G.nodes[n]['out_weight'] = sum(d['weight'] for _, _, d in G.out_edges(n, data=True))
        G.nodes[n]['in_weight']  = sum(d['weight'] for _, _, d in G.in_edges(n, data=True))

    graphs[window] = G

print(f'Built {len(graphs)} directed SVO graph(s)')

## Step 4: Validation report — agents, targets, top relations

In [ ]:
for window, G in graphs.items():
    connected = [n for n in G.nodes() if G.degree(n) > 0]   # in + out for a DiGraph
    total_w = sum(d['weight'] for _, _, d in G.edges(data=True))
    print(f'\n========== {window} ==========')
    print(f'  Actors with a relation : {len(connected)}/{G.number_of_nodes()}')
    print(f'  Directed edges         : {G.number_of_edges()}')
    print(f'  Total edge weight      : {total_w}')

    out_hubs = sorted(connected, key=lambda n: -G.nodes[n]['out_weight'])[:3]
    in_hubs  = sorted(connected, key=lambda n: -G.nodes[n]['in_weight'])[:3]
    print(f'  Top agents  (out-weight): {[(n, G.nodes[n]["out_weight"]) for n in out_hubs]}')
    print(f'  Top targets (in-weight) : {[(n, G.nodes[n]["in_weight"]) for n in in_hubs]}')

    print('  Top directed relations:')
    for u, v, d in sorted(G.edges(data=True), key=lambda x: -x[2]['weight'])[:5]:
        print(f'    {d["weight"]:4d}  [{d["dominant_polarity"]:>8}]  {u} -> {v}  '
              f'({d["top_verb"]}, {d["dominant_cameo"]})')

## Step 5: Save GraphML

`G_svo_{window}.graphml` — a directed graph readable by Gephi, Cytoscape, igraph, etc. All node/edge attributes are scalar (the `verbs`/`sources` dicts from 05b are collapsed to `top_verb`/`top_source` + counts, since GraphML cannot store dicts).

In [ ]:
for window, G in graphs.items():
    out = NETWORKS_DIR / 'svo' / 'graphml' / f'{window_file_label(window)}_svo.graphml'
    nx.write_graphml(G, out)
    print(f'GraphML : {out.name}')

## Step 6: Save GEXF (Gephi preferred)

In [ ]:
for window, G in graphs.items():
    out = NETWORKS_DIR / 'svo' / 'gexf' / f'{window_file_label(window)}_svo.gexf'
    nx.write_gexf(G, out)
    print(f'GEXF    : {out.name}')

## Step 7: pyvis interactive HTML (directed)

Directed arrows show `subject → object`; edge colour = `dominant_polarity` (green positive / red negative / dark-grey mixed — SVO/CAMEO has no neutral quadrant), width = weight, node size = total (in+out) degree. Only connected actors are rendered to keep the sparse graph legible. Hover an edge for its CAMEO type and top verb.

In [ ]:
try:
    from pyvis.network import Network
    HAS_PYVIS = True
except ImportError:
    HAS_PYVIS = False
    print('pyvis not installed — skip this cell or run: pip install pyvis')

ACTOR_COLOR = '#4c72b0'

def polarity_legend_html(items):
    """Floating HTML legend (pyvis has no native legend support)."""
    rows = ''.join(
        f'<div style="display:flex;align-items:center;margin:2px 0;">'
        f'<span style="width:14px;height:14px;background:{color};display:inline-block;'
        f'margin-right:6px;border-radius:2px;"></span><span>{label}</span></div>'
        for label, color in items
    )
    return (
        '<div style="position:fixed;top:12px;right:12px;z-index:999;'
        'background:rgba(255,255,255,0.95);border:1px solid #ccc;border-radius:6px;'
        'padding:8px 10px;font-family:sans-serif;font-size:12px;'
        'box-shadow:0 1px 4px rgba(0,0,0,0.2);">'
        '<div style="font-weight:bold;margin-bottom:4px;">Edge polarity (SVO)</div>'
        + rows + '</div>'
    )

def inject_legend(html_path, items):
    html = html_path.read_text(encoding='utf-8')
    html_path.write_text(
        html.replace('</body>', polarity_legend_html(items) + '</body>', 1),
        encoding='utf-8',
    )

SVO_LEGEND = [('positive', POLARITY_COLOR['positive']),
              ('negative', POLARITY_COLOR['negative']),
              ('mixed',    POLARITY_COLOR['mixed'])]
OPTIONS = '{"physics": {"forceAtlas2Based": {"gravitationalConstant": -70, "springLength": 130, "avoidOverlap": 0.9}, "solver": "forceAtlas2Based", "minVelocity": 0.75, "timestep": 0.4}, "interaction": {"hover": true, "navigationButtons": true, "keyboard": true}}'

if HAS_PYVIS:
    for window, G in graphs.items():
        connected = [n for n in G.nodes() if G.degree(n) > 0]
        Gsub = G.subgraph(connected)
        net = Network(height='720px', width='100%', bgcolor='#ffffff',
                      font_color='#222222', notebook=False, directed=True,
                      cdn_resources='in_line')
        max_w = max((d['weight'] for _, _, d in Gsub.edges(data=True)), default=1) or 1
        for n in Gsub.nodes():
            d = G.nodes[n]
            net.add_node(
                n, label=n, color=ACTOR_COLOR, shape='dot',
                size=10 + 4 * (d['out_degree'] + d['in_degree']),
                title=(f"{n}<br>entity_type: {d.get('entity_type', '')}<br>"
                       f"agent (out): {d['out_degree']} edges / {d['out_weight']} wt<br>"
                       f"target (in): {d['in_degree']} edges / {d['in_weight']} wt"),
                physics=True,
            )
        for u, v, d in Gsub.edges(data=True):
            net.add_edge(
                u, v, value=d['weight'], width=1 + 6 * (d['weight'] / max_w),
                color=POLARITY_COLOR.get(d['dominant_polarity'], '#9AA0A6'),
                title=(f"{u} to {v}<br>weight: {d['weight']}<br>"
                       f"CAMEO: {d['dominant_cameo']}<br>polarity: {d['dominant_polarity']}<br>"
                       f"top verb: {d['top_verb']}"),
            )
        net.set_options(OPTIONS)
        out = NETWORKS_DIR / 'svo' / 'html' / f'{window_file_label(window)}_svo.html'
        net.write_html(str(out), open_browser=False, notebook=False)
        inject_legend(out, SVO_LEGEND)
        print(f'HTML    : {out.name}')

    print('\nDirected SVO previews written — arrows = subject->object; '
          'edge colour = polarity; node size = in+out degree.')